# MCP in Python

![](https://mintcdn.com/mcp/bEUxYpZqie0DsluH/images/mcp-simple-diagram.png?w=1650&fit=max&auto=format&n=bEUxYpZqie0DsluH&q=85&s=a131a609c7b6a70f342f493bbad57fcb)

- https://modelcontextprotocol.io/
- https://modelcontextprotocol.io/docs/develop/build-server

* **Model Context Protocol (MCP)** is a standard protocol enabling AI models (like Claude) to interact with external tools, services, and data sources.
* It defines a communication framework based on **JSON-RPC over stdio or HTTP**.
* Solves the problem of custom tool integrations by introducing a **standard, declarative interface**.
* Designed for security, modularity, and composability.

**Roles in MCP:**

* **Host**: The orchestrator (e.g., Claude, or a LangChain agent) that manages conversation context and tool availability.
* **Client**: A connector that handles communication with tool servers.
* **Server**: Exposes tools (Python functions) and implements logic behind them.
|

In [1]:
# pip install mcp fastmcp
import sys, os
from fastmcp import FastMCP

In [2]:
mcp = FastMCP()

In [3]:
@mcp.tool
def evaluate_expression(expression: str) -> float:
    """Evaluates a basic arithmetic expression."""
    return eval(expression)

In [4]:
def calculate_tax_irs(income):
    # Define the tax brackets and rates
    brackets = [(0, 20550, 0.10), (20551, 83550, 0.12), (83551, 178150, 0.22),
                (178151, 340100, 0.24), (340101, 431900, 0.32),
                (431901, 647850, 0.35), (647851, float('inf'), 0.37)]
    
    tax = 0
    for lower, upper, rate in brackets:
        if income > upper:
            tax += (upper - lower) * rate
        else:
            tax += (income - lower) * rate
            break

    return tax

In [5]:
@mcp.tool
def tax_resident(income: float) -> float:
    """Calculate the tax liability of residents. Input: income as a number."""
    return calculate_tax_irs(income)

In [6]:
@mcp.tool
def tax_nonresident(income: float) -> float:
    """Calculate the tax liability of nonresidents. Input: income as a number."""
    return income * 0.25

In [7]:
@mcp.resource('config://tax_residency_rule')
def tax_residency_rules():
    return 'If you are not a U.S. citizen, you are considered a nonresident of the United States for U.S. tax purposes unless you meet one of two tests. You are a resident  of the United States for tax purposes if you meet either the green card test or the substantial presence test for the calendar year (January 1 – December 31).'


In [8]:
# test if the script is running as a standalone program or a notebook
if __name__ == '__main__' and '__file__' in globals():
    print("Running as a script")
    interactive = False
    mcp.run()
    # additional testing here ...
    print(evaluate_expression("1+2"))
else:
    print("Running as a notebook")
    interactive = True

Running as a notebook


In [9]:
if not interactive:
    sys.exit(1)

Additional notebook code below (to be ignored when running as a script)

In [10]:
evaluate_expression("1+2")

3